# Cardiac CNN-JEPA — pré-entraînement sur Colab

Variante **encodeur CNN 1D** du JEPA (dérivations = canaux, 40 tokens temporels),
à côté du ViT. Config `jepa/configs/cnn.yaml`, run séparé `runs/cnn_v1` — n'écrase
**rien** du ViT (`tiny_v1`).

**Avant de lancer** : `Exécution ▸ Modifier le type d'exécution ▸ GPU`.

Idempotent : re-lance après déconnexion, `--resume auto` reprend où ça s'était arrêté.
Renseigne `REPO_URL` dans la cellule de configuration.

## 1. Vérifier le GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch; print("torch", torch.__version__, "| cuda dispo :", torch.cuda.is_available())

## 2. Monter Drive et configurer les chemins

Même Drive/cache que le notebook ViT ; seul le dossier de run change (`cnn_v1` au lieu de `tiny_v1`).

In [ ]:
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# >>> RENSEIGNE TON URL DE REPO ICI (pas de chevrons !) <<<
# exemple : "https://github.com/moncompte/cardiac-jepa.git"
REPO_URL = "https://github.com/JulesV19/cardiac-JEPA"

if not REPO_URL:
    raise ValueError(
        "REPO_URL est vide. Mets l'URL de ton repo GitHub dans cette cellule, "
        "par ex. https://github.com/moncompte/cardiac-jepa.git")

DRIVE_ROOT  = Path('/content/drive/MyDrive/cardiac-jepa')   # persiste entre sessions
DATA_DRIVE  = DRIVE_ROOT / 'data'                            # les 2 CSV (petits)
CACHE_DRIVE = DRIVE_ROOT / 'ptbxl_100hz.npy'                 # cache signaux (1,05 Go)
CACHE_LOCAL = Path('/content/ptbxl_100hz.npy')               # copie sur disque local = rapide
CNN_RUN     = 'cnn_v2'                                       # <- change ici : 'cnn_v1' ou 'cnn_v2'
CNN_CONFIG  = f'jepa/configs/{CNN_RUN}.yaml'
RUN_CNN     = DRIVE_ROOT / 'runs' / CNN_RUN                  # checkpoints + metrics.csv (CNN)
REPO_DIR    = Path('/content/cardiac-jepa')

RUN_CNN.mkdir(parents=True, exist_ok=True)
print("Drive prêt :", DRIVE_ROOT)

## 3. Cloner / mettre à jour le code

Tu édites en local, `git push`, puis ré-exécute cette cellule.

In [ ]:
import subprocess

if REPO_DIR.exists():
    r = subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
                       capture_output=True, text=True)
else:
    r = subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)],
                       capture_output=True, text=True)
print(r.stdout or r.stderr)
if r.returncode != 0:
    raise RuntimeError(
        f"git a échoué (code {r.returncode}). Vérifie REPO_URL, que le repo existe, "
        "et qu'il est public (sinon il faut un token d'accès).")

assert (REPO_DIR / "requirements.txt").exists(), \
    f"{REPO_DIR}/requirements.txt introuvable — le clone a-t-il vraiment réussi ?"

%cd {REPO_DIR}
!pip install -q -r requirements.txt
print("code prêt :", REPO_DIR)

## 4. Données (déposées manuellement sur Drive)

Le cache est construit en local et déposé sur Drive — **rien à télécharger ici**.
Arborescence attendue :

```
MyDrive/cardiac-jepa/
├── ptbxl_100hz.npy        (1,05 Go)  cache des signaux 100 Hz
└── data/
    ├── ptbxl_database.csv (6,6 Mo)
    └── scp_statements.csv (10 Ko)
```

Pourquoi un cache plutôt que les fichiers bruts : `records100` contient **~43 000 petits
fichiers**, atroces à uploader sur Drive *et* atroces à lire depuis un Drive monté.
Un seul fichier contigu, copié sur le disque local de Colab, c'est instantané.

L'ordre des lignes de `ptbxl_database.csv` définit l'index du cache : **ces deux fichiers
doivent aller ensemble.**

In [ ]:
import shutil, time
import numpy as np

required = [CACHE_DRIVE,
            DATA_DRIVE / 'ptbxl_database.csv',
            DATA_DRIVE / 'scp_statements.csv']
missing = [p for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Fichiers manquants sur Drive :\n  " + "\n  ".join(str(m) for m in missing) +
        "\n\nDépose-les exactement à ces emplacements, puis relance cette cellule.")

if not CACHE_LOCAL.exists():
    print("Copie du cache Drive -> disque local (~1 Go, une fois par session)...")
    t0 = time.time()
    shutil.copy(CACHE_DRIVE, CACHE_LOCAL)
    print(f"copié en {time.time()-t0:.0f}s")

# Le code lit ces variables d'environnement (voir jepa/data.py).
os.environ['PTBXL_DATA_DIR'] = str(DATA_DRIVE)   # CSV uniquement
os.environ['PTBXL_CACHE']    = str(CACHE_LOCAL)

a = np.load(CACHE_LOCAL, mmap_mode='r')
assert a.shape[1:] == (1000, 12), f"forme inattendue: {a.shape}"
print(f"cache OK : {a.shape} {a.dtype}")

## 5. Sanity check : les données se chargent

In [ ]:
from jepa.data import PTBXLDataset
ds = PTBXLDataset('pretrain')
x = ds[0]
print(f"pretrain={len(ds)} ECG | un échantillon: {tuple(x.shape)} "
      f"| moyenne={x.mean():.3f} std={x.std():.3f}  (z-normé par dérivation)")

## 6. Entraînement CNN-JEPA

`--resume auto` reprend depuis `latest.pt`. Déconnexion Colab -> ré-exécute.

In [ ]:
!python -m jepa.train \
    --config "{CNN_CONFIG}" \
    --out "{RUN_CNN}" \
    --resume auto \
    --workers 2

## 7. Courbes de collapse

Mêmes critères que le ViT : `emb_std_ctx` ne tend pas vers 0, `eff_rank_ctx`
reste élevé, `jepa` baisse progressivement. (rang effectif max = embed_dim = 192)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

m = pd.read_csv(RUN_CNN / 'metrics.csv')
tr, va = m[m.phase == 'train'], m[m.phase == 'val']

fig, ax = plt.subplots(1, 4, figsize=(19, 4))

# LE graphe qui compte : qualité de prédiction, insensible à la difficulté des cibles.
ax[0].plot(va.epoch, va.r2, marker='o', color='#C44E52', label='R²')
ax[0].axhline(0, ls='--', c='k', lw=1, label='prédicteur trivial (moyenne)')
ax[0].set_title('R² — apprend-on vraiment ?'); ax[0].set_xlabel('epoch'); ax[0].legend()

ax[1].plot(tr.step, tr.jepa, lw=0.8)
ax[1].set_title('L_jepa (cible mouvante : non comparable seule)'); ax[1].set_xlabel('step')

ax[2].plot(va.epoch, va.emb_std_ctx, label='contexte')
ax[2].plot(va.epoch, va.emb_std_tgt, label='cible')
ax[2].plot(va.epoch, va.pred_std, label='prédiction')
ax[2].axhline(0.1, ls='--', c='r', lw=1, label='seuil collapse')
ax[2].set_title('écart-type des descriptions'); ax[2].set_xlabel('epoch'); ax[2].legend()

ax[3].plot(va.epoch, va.eff_rank_ctx, label='contexte')
ax[3].plot(va.epoch, va.eff_rank_tgt, label='cible')
ax[3].set_title('rang effectif (max 192)'); ax[3].set_xlabel('epoch'); ax[3].legend()

plt.tight_layout(); plt.show()
print(va[['epoch','r2','cos','emb_std_ctx','eff_rank_ctx','eff_rank_tgt']].tail(10).to_string(index=False))

## 8. Sonde linéaire — CNN-JEPA vs CNN **aléatoire de même architecture**

Le contrôle utilise `--random-init --ckpt` : mêmes dimensions CNN, poids
aléatoires (comparaison iso-architecture, pas un ViT-tiny).

In [ ]:
from pathlib import Path
import json

CKPT = RUN_CNN / 'ckpt_e99.pt'
assert CKPT.exists(), f"{CKPT} absent — liste: {sorted(p.name for p in RUN_CNN.glob('*.pt'))}"

# CNN-JEPA pré-entraîné
!python -m jepa.probe --ckpt "{CKPT}" --encoder target --workers 2 --out "{RUN_CNN}/probe_cnn.json"
# CNN ALÉATOIRE de même archi (iso-architecture)
!python -m jepa.probe --random-init --ckpt "{CKPT}" --workers 2 --out "{RUN_CNN}/probe_random.json"

j = json.load(open(RUN_CNN / 'probe_cnn.json'))
r = json.load(open(RUN_CNN / 'probe_random.json'))
print(f"CNN-JEPA    macro-AUROC TEST : {j['test_macro_auroc']:.4f}")
print(f"CNN random  macro-AUROC TEST : {r['test_macro_auroc']:.4f}")
print(f"écart JEPA - random         : {j['test_macro_auroc'] - r['test_macro_auroc']:+.4f}")

## 9. Phase 2 — fine-tuning (CNN dégelé)

Encodeur CNN **dégelé**, `lr` 10× plus faible que la tête, sélection du meilleur epoch
sur fold 9, test sur fold 10 une seule fois. Le contrôle est un CNN **de même archi**
mais aléatoire (`--random-init --ckpt`) — pas un ViT. C'est le régime où le CNN est
censé atteindre son plafond (~0,93 supervisé).

In [ ]:
CKPT = RUN_CNN / 'ckpt_e99.pt'
assert CKPT.exists(), sorted(p.name for p in RUN_CNN.glob('*.pt'))
CLF_CNN_RANDOM = DRIVE_ROOT / 'runs' / f'clf_{CNN_RUN}_random'
# contrôle ISO-ARCHI : même CNN, poids aléatoires
!python -m jepa.classify --random-init --ckpt "{CKPT}" --out "{CLF_CNN_RANDOM}" --workers 2

In [ ]:
CLF_CNN_JEPA = DRIVE_ROOT / 'runs' / f'clf_{CNN_RUN}_jepa'
!python -m jepa.classify --ckpt "{CKPT}" --encoder target --out "{CLF_CNN_JEPA}" --workers 2

## 10. Régime peu-de-labels

Fine-tuning JEPA vs aléatoire (iso-archi) sur 1 %, 5 %, 10 %, 100 % des labels.
Sous-échantillon identique entre bras (même graine), val/test complets. Petites
fractions rejouées sur 5 graines (écart *pairé*). Idempotent : relançable après
déconnexion. Dossier séparé `lowlabel_cnn` (n'écrase pas le sweep ViT).

In [ ]:
import subprocess, sys
from pathlib import Path

CKPT = RUN_CNN / 'ckpt_e99.pt'
assert CKPT.exists(), sorted(p.name for p in RUN_CNN.glob('*.pt'))
SWEEP = DRIVE_ROOT / 'runs' / f'lowlabel_{CNN_RUN}'

SEEDS_BY_FRAC = {0.01: [0, 1, 2, 3, 4],
                 0.05: [0, 1, 2, 3, 4],
                 0.10: [0],
                 1.00: [0]}
FRACS = sorted(SEEDS_BY_FRAC)

def run(arm, frac, seed):
    out = SWEEP / f'{arm}_f{frac}_s{seed}'
    if (out / 'result.json').exists():
        print(f'  {arm} frac={frac} seed={seed} : déjà fait'); return
    # random ISO-ARCHI : --random-init AVEC --ckpt (même CNN, poids aléatoires)
    src = (['--random-init', '--ckpt', str(CKPT)] if arm == 'random'
           else ['--ckpt', str(CKPT), '--encoder', 'target'])
    cmd = [sys.executable, '-m', 'jepa.classify', *src, '--out', str(out),
           '--train-frac', str(frac), '--seed', str(seed), '--workers', '2']
    print(f'\n=== {arm}  frac={frac}  seed={seed} ===', flush=True)
    assert subprocess.run(cmd).returncode == 0, f'échec {arm} frac={frac} seed={seed}'

for frac in FRACS:
    for seed in SEEDS_BY_FRAC[frac]:
        for arm in ('random', 'jepa'):
            run(arm, frac, seed)
print('\nsweep terminé')

### Courbe peu-de-labels

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt

rows = []
for frac in FRACS:
    for seed in SEEDS_BY_FRAC[frac]:
        res = {}
        for arm in ('random', 'jepa'):
            p = SWEEP / f'{arm}_f{frac}_s{seed}' / 'result.json'
            if p.exists():
                res[arm] = json.load(open(p))['test_macro_auroc']
        if len(res) == 2:
            rows.append({'frac': frac, 'seed': seed, 'random': res['random'],
                         'jepa': res['jepa'], 'gap': res['jepa'] - res['random']})
d = pd.DataFrame(rows)

# Écart PAIRÉ par graine (même sous-échantillon entre bras) = le bon estimateur de l'effet.
def agg(g):
    return pd.Series({'random': g.random.mean(), 'jepa': g.jepa.mean(),
                      'écart': g.gap.mean(),
                      'écart_std': g.gap.std(ddof=1) if len(g) > 1 else 0.0,
                      'n': len(g)})
tab = d.groupby('frac').apply(agg)
print(tab.round(4).to_string())

fig, ax = plt.subplots(figsize=(7, 5))
for arm, c in [('jepa', '#C44E52'), ('random', '#4C72B0')]:
    m = d.groupby('frac')[arm].mean()
    s = d.groupby('frac')[arm].std(ddof=1).fillna(0.0)
    ax.errorbar(m.index * 100, m.values, yerr=s.values, marker='o', capsize=3,
                color=c, label=arm)
ax.set_xscale('log')
ax.set_xlabel("% des labels d'entraînement (échelle log)")
ax.set_ylabel('macro-AUROC (test)')
ax.set_title('Le pré-entraînement JEPA aide-t-il quand les labels manquent ?')
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()